## Fine-tuning a model to generate job descriptions from titles

In [16]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes trl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
# Import libraries
import pandas as pd
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

In [18]:
# Load dataset
df = pd.read_csv('/kaggle/input/linkedin-job-postings/postings.csv')[0:500]
df = df[['title', 'description']].dropna()
df = df[df['description'].str.len() > 100]  # filter short examples

# Format into prompt-response pairs
def format_example(row):
    return {
        "prompt": f"Generate a job description for the title: {row['title']}",
        "completion": row["description"]
    }

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_example)
dataset = dataset.train_test_split(test_size=0.05)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [19]:
# Tokenization and prompt formatting
from transformers import AutoTokenizer

model_id = "NousResearch/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def apply_prompt_format(example):
    return {
        "text": f"### Instruction:\n{example['prompt']}\n\n### Response:\n{example['completion']}"
    }

tokenized_dataset = dataset.map(apply_prompt_format).map(
    lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=512),
    batched=True
)

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

In [20]:
# Load model and prepare LoRA fine-tuning
from transformers import AutoModelForCausalLM, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True
)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [21]:
# Training configuration
training_args = SFTConfig(
    output_dir="./job-desc-model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    max_seq_length=512,
    weight_decay=0.01,
    report_to="none"
)

average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.


In [22]:
# Start fine-tuning
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
)

trainer.train()

Truncating train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss
1,1.616900,1.670863
2,1.565400,1.639381
3,1.488900,1.641223


TrainOutput(global_step=714, training_loss=1.76849249564633, metrics={'train_runtime': 3612.7977, 'train_samples_per_second': 0.394, 'train_steps_per_second': 0.198, 'total_flos': 2.90711952949248e+16, 'train_loss': 1.76849249564633})

In [23]:
# Inference (test generation)
def generate_job_description(title: str):
    prompt = f"### Instruction:\nGenerate a job description for the title: {title}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=300, do_sample=True, top_p=0.95, temperature=0.8)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_job_description("AI Research Scientist"))

### Instruction:
Generate a job description for the title: AI Research Scientist

### Response:
Job Title: AI Research Scientist
Type: Full-time
Location: Mumbai, India

Job Description:
Our AI Research Scientists will work under the guidance of experienced AI research scientists to develop novel AI applications and techniques to solve real-world problems. You will have the opportunity to work on a variety of projects from AI for healthcare to AI for supply chain management and everything in between. We have an agile development process where you will work closely with stakeholders to gather requirements and develop AI solutions that meet the needs of our clients. You will also have the opportunity to collaborate with other departments within the company to ensure that our AI solutions are integrated into our products and services.
Responsibilities:
Design, develop, and implement novel AI applications and techniques to solve real-world problems.Work closely with stakeholders to gather 

In [24]:
# Save Final Checkpoint
model.save_pretrained("./job-desc-model/final_adapter")
tokenizer.save_pretrained("./job-desc-model/final_adapter")

('./job-desc-model/final_adapter/tokenizer_config.json',
 './job-desc-model/final_adapter/special_tokens_map.json',
 './job-desc-model/final_adapter/tokenizer.model',
 './job-desc-model/final_adapter/added_tokens.json',
 './job-desc-model/final_adapter/tokenizer.json')

In [25]:
# Load Trained Model Later
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./job-desc-model/final_adapter")

# Load base model and merge with LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    "NousResearch/Llama-2-7b-chat-hf",
    device_map="auto",
    load_in_4bit=True
)

model = PeftModel.from_pretrained(base_model, "./job-desc-model/final_adapter")


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [26]:
# Merge LoRA with Base Model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./job-desc-model/full_merged")
tokenizer.save_pretrained("./job-desc-model/full_merged")

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('./job-desc-model/full_merged/tokenizer_config.json',
 './job-desc-model/full_merged/special_tokens_map.json',
 './job-desc-model/full_merged/tokenizer.model',
 './job-desc-model/full_merged/added_tokens.json',
 './job-desc-model/full_merged/tokenizer.json')

In [27]:
# Zip the directory
!zip -r job-desc-model.zip ./job-desc-model/final_adapter

  adding: job-desc-model/final_adapter/ (stored 0%)
  adding: job-desc-model/final_adapter/tokenizer_config.json (deflated 72%)
  adding: job-desc-model/final_adapter/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 85%)
  adding: job-desc-model/final_adapter/adapter_config.json (deflated 54%)
  adding: job-desc-model/final_adapter/added_tokens.json (stored 0%)
  adding: job-desc-model/final_adapter/adapter_model.safetensors (deflated 7%)
  adding: job-desc-model/final_adapter/tokenizer.model (deflated 55%)
  adding: job-desc-model/final_adapter/special_tokens_map.json (deflated 72%)
  adding: job-desc-model/final_adapter/README.md (deflated 66%)
